## Chapter 1: Data Collection

Before we can fit a surrogate model or run Bayesian Optimization, we need data.
This chapter collects that data in three simple steps:
1. Sample 128 random-but-well-spread recipes (Sobol sampling)
2. Test all of them on all 3 scales (micro, bench, pilot)
3. Pick 12 interesting recipes and repeat them 5x to measure noise

In [ ]:
import numpy as np
import pandas as pd
import time
from scipy.stats import qmc
from client import BioreactorClient, USER, PASSWORD, BASE_URL

# The 5 parameters we can control, and their min/max allowed values
param_names = ["T", "pH", "F1", "F2", "F3"]
lower_bounds = [20.0, 3.0, 0.0, 0.0, 0.0]
upper_bounds = [60.0, 9.5, 2.0, 2.0, 2.0]

scales = ["micro", "bench", "pilot"]

n_recipes = 128   # how many different recipes to try
n_repeats = 5     # how many times to repeat each chosen recipe

### Step 1: Generate 128 recipes with Sobol sampling

A Sobol sequence is just a smart way of picking "random" points that
spread out evenly across the space, instead of clumping together like
normal random numbers would. This gives us good coverage of all 5 parameters.

In [ ]:
# Sobol sampler needs n_recipes to be a power of 2 (128 = 2^7)
sampler = qmc.Sobol(d=len(param_names), scramble=True)
unit_samples = sampler.random_base2(m=7)  # 2^7 = 128 points, values between 0 and 1

# Stretch the 0-1 values out to our real parameter ranges
scaled_samples = qmc.scale(unit_samples, lower_bounds, upper_bounds)

recipes = pd.DataFrame(scaled_samples, columns=param_names)
recipes.head()

### Step 2: Connect to the simulator

In [ ]:
client = BioreactorClient(BASE_URL)
client.login(USER, PASSWORD)
print("Connected!")

### Step 3: Run all 128 recipes on all 3 scales

This is our "exploration" data — a broad first look at how yield behaves
across the whole parameter space, at every scale.

In [ ]:
exploration_results = []

for recipe_id in range(n_recipes):
    recipe = recipes.loc[recipe_id]
    
    for scale in scales:
        result = client.run(
            scale=scale,
            T=recipe["T"], pH=recipe["pH"],
            F1=recipe["F1"], F2=recipe["F2"], F3=recipe["F3"]
        )
        
        exploration_results.append({
            "recipe_id": recipe_id,
            "scale": scale,
            "T": recipe["T"], "pH": recipe["pH"],
            "F1": recipe["F1"], "F2": recipe["F2"], "F3": recipe["F3"],
            "yield": result["Y"],
            "cost": result["cost_eur"]
        })
        
        time.sleep(0.1)  # be nice to the server

    if recipe_id % 20 == 0:
        print(f"Done with recipe {recipe_id}/{n_recipes}")

df_exploration = pd.DataFrame(exploration_results)
print(f"Collected {len(df_exploration)} data points.")

### Step 4: Pick 12 recipes to repeat

We want a mix: some recipes that gave low yield, some medium, some high.
We use the pilot-scale results to sort recipes by yield, then take 4 from
each third of that list.

In [ ]:
# Only look at pilot-scale results, sorted worst to best yield
pilot_results = df_exploration[df_exploration["scale"] == "pilot"]
pilot_results = pilot_results.sort_values("yield").reset_index(drop=True)

n = len(pilot_results)
low_group = pilot_results.iloc[0 : n//3]
mid_group = pilot_results.iloc[n//3 : 2*n//3]
high_group = pilot_results.iloc[2*n//3 : n]

# Take 4 random recipes from each group
chosen_low = low_group.sample(4, random_state=42)
chosen_mid = mid_group.sample(4, random_state=42)
chosen_high = high_group.sample(4, random_state=42)

chosen_recipe_ids = pd.concat([chosen_low, chosen_mid, chosen_high])["recipe_id"].values
print("Recipes we'll repeat:", chosen_recipe_ids)

### Step 5: Repeat the 12 chosen recipes 5 times each

Running the same recipe multiple times tells us how noisy the simulator's
measurements are — same input, but the output will vary a bit each time.

In [ ]:
repeat_results = []

for repeat_number in range(1, n_repeats + 1):
    for recipe_id in chosen_recipe_ids:
        recipe = recipes.loc[recipe_id]
        
        for scale in scales:
            result = client.run(
                scale=scale,
                T=recipe["T"], pH=recipe["pH"],
                F1=recipe["F1"], F2=recipe["F2"], F3=recipe["F3"]
            )
            
            repeat_results.append({
                "recipe_id": recipe_id,
                "scale": scale,
                "repeat": repeat_number,
                "T": recipe["T"], "pH": recipe["pH"],
                "F1": recipe["F1"], "F2": recipe["F2"], "F3": recipe["F3"],
                "yield": result["Y"],
                "cost": result["cost_eur"]
            })
            
            time.sleep(0.1)

    print(f"Finished repeat {repeat_number}/{n_repeats}")

df_repeats = pd.DataFrame(repeat_results)

### Step 6: Save everything

Combine the exploration data and the repeat data into one file, so we
don't have to re-run the simulator every time we reopen this notebook.

In [ ]:
df_exploration["phase"] = "exploration"
df_repeats["phase"] = "repeat"

df_all = pd.concat([df_exploration, df_repeats], ignore_index=True)
df_all.to_csv("task1_2_data.csv", index=False)

print(f"Saved {len(df_all)} total rows to task1_2_data.csv")